ESEMPIO: USARE HAGGING FACE TRANSFORMERS PER TEXT GENERATION

Hugging Face permette di usare l' ecosistema Transformers, generando testo in pochi secondi, imparando a dominare la creatività delle macchine attraverso parametri precisi.

Huggin Face Transformers è uno dei modi più pratici e veloci per usare modelli già addestrati per fare text generation, cioè generazione automatica di testo.

prompt -> tokenizer -> token ID -> Transformer -> probabilità del prossimo token -> scelta del token -> il token viene aggiunto al testo -> si ripete

Un LLM genera testo in modo autoregressivo: predice un token alla volta usando com econtesto tutto ciò che ha visto prima. E' la previsoine del token successivo dato un testo iniziale e i token generati fino a quel momento.

transformes è un libreria di Python che permette di usare modelli Transformer già addestrati come: GPT, Llama, Qwen, Mistral, Gemma, T5, BART, ...

Non devi necessariamente addestrare un modello puoi fare:
modello già addestrato -> download -> prompt -> generazione

Il modello viene generalmente scaricato da Hugging Face Hub e poi memorizzato in cache sul PC.

*** PIPELINE () ***
(il metodo più semplice)

Hugging Face mette a disposizione pipeline, che nasconde gran parte della complessità. E' l'API di inferenza più semplice della libreria.
La pipeline fa automaticamente diverse cose:
caricamento tokenizer + caricamento modello + tokenizzazione + inferenza + generazione + decodifica
Tu devi fare solo: generator(prompt)
Pipeline incapsula l'intera catenda di elaborazione: dal testo grezzo alla risposta finale.
Questa astrazione permette di cambiare il modello sottostante o il compito specifico con una singola riga di codice.
Incapsula tutta la compressità del preprocessing e del decoding in un'unica inerfaccia.
Sotto il cofano c'è un'automazione incredibile.
L'automazione è totale, la pipeline si occupa di scaricare il modello, configurare il tokenizer, e spostare i dati sulla GPU e tradurre i risultati in testo leggibile.
Questo rende l'AI accessibile a tanti

Cosa succede dietro a pipeline?
esempio:
prompt: "Artificial intelligence will"
il tokenizer potrebbe trasformare il prompt in qualcosa simile a:
Artificial -> 8001
intelligence -> 4430
will -> 481
quindi il modello riceve quei numeri e non le parole. [8001,4430,481]
produce una distribuzione di probabilità sul proprio vocabolario
esempio:
transform       0.18
change          0.16
be              0.12
help            0.08
destroy         0.02    
banana          0.0001
...
il modello sceglie un token (supponiamo change), aggiunte il token scelto alla frase e riparte
"Artificial intelligence will"
"Artificial intelligence will change"
"Artificial intelligence will change the"
"Artificial intelligence will change the way"
...

Il modello pipeline è a caricamento Lazy, non occupa spazio finchè non ne hai bisogno, modello e tokenizer vengono scaricati e istanziati solo alla prima chiamata, ottimizzando le risorse di rete.
La gestione del device base un parametro di dire di usare la scheda video e la pipeline si occupa di tutto.
Senza parametri aggiuntivi, la pipeline utilizza configurazioni conservative che garantiscono stabilità a scapito della varietà creativa, pensato per dare risposte stabili.

La pipeline è compata da 3 componenti: 1) tokenizer 2) model 3) post-processor.
Il flusso diventa lineare: il testo (prompt) viene spezzettato e numerato, passa attraverso i layer di attenzione della rete, ed infine i risultati numerici vengono ripuliti e trasformato, di nuovo, in parole umane 

Questa scomposizione è fondamentale, perchè quando le cose non vanno come previsto, è necessario capire chi sbalgia.

Il tokenizer trasforma il linguaggio in matematica.
I modelli Transformer non leggono stringhe, ma sequenze di numeri. Il tokenizer ha il compito critico di mappare linguaggio umano in ID numerici.
Il tokenizer è quindi il nostro traduttore simultaneo. Il suo compito è mappare le stringhe di testo in ID numerici.
Ma non è una semplice assegnazione 1 a 1, gli algoritmi moderni utilizzano diverse tecniche per coprere la vastita del linguaggio umano., senza esplodere in memoria.
Questi algoritmi riscono a gestire parole magari mai visto prima

Algoritmi di SubWord (sottoparole)
Algoritmi come Byte-Pair Encoding (BPE) non imparano intere parole, ma pezzi di esse.
Se il modello incontra una parola nuova la scompone nella sua radice e nei suoi suffissi
In questo modo il vocabolario rimane intatto.


** AutoModelForCausalLM **
AutoTokenizer.from_pretrained(model_name)
(più controllabile)

Huggin Face offre due livelli: 1) pipeline: il più semplice ma meno controllabile 2) Autotokenizer + AutoModelForCausalLM: più controllabile

* Partiamo da Autotokenizer
Si chiama Autotokenizer e non semplicemente Tokenizer perchè modelli diversi utilizzano tokenizer differenti.
GPT-2, Llama, Qwen, Mistral utilizzano tokenizer diversi.
Con AutoTokenizer.from_pretrained(model_name)
Hagging Face legge la configurazione del modello e sceglie automaticamente il tokenizer corretto.

* Carichiamo il modello
model = AutoModelForCausalLM.from_pretrained(model_name)
CausalLM sono modelli progettati per fare qualcosa del tipo:
token1, token2, token3 ... token(n) -> token(n+1)
GPT; Llama e Qwen appartengono a questa categoria

Perchè 'casual'
Perchè durante la previsione del token corrente il modello può gardare solo il passato e non il futuro.
Durante il training viene usata una 'causal mask' che impedisce di 'sbirciare' i token futuri.

** Generazione **
model.generate()
Ora possiamo generare.
generate() è l'API centrale per generare testo.
Il risultato perè non è ancora testo, outputs contiene token ID, quindi bisogna decodificare (token. decode())

** max_new_tokens **
e' uno dei parametri più importanti, significa: genera al massimo altri n token dopo il prompt. Non significa esattamente n parole, perchè token e parole non coincidono

Come scegliere il prossimo token?
esistono diverse strategie

- do_sample -
La strategia più semplice è scegliere sempre il token con probabilità maggiore
outputs = model.generate(**inputs,max_new_tokens=50,do_sample=False)
Vantaggio: più deterministico, svantaggio: poco creativo e spesso ripetitivo

- sampling -
non prendere sempre il primo, estrai un token secondo le probabilità
do_sample=True
Quindi non prende sempre il token con la probabilità più alta ma tra quelli che hanno la probabilità più alta. Questo produce maggiore varietà

- temperature
la temperatura modifica la distribuzione delle probabilità prima del campionamento. Hugging Face le espone come parametro di generazione. Il valore base è 1.0
temperatura più  bassa -> più conservativo 
temperatura più alta -> più casuale
0.1 - 0.3 - molto deterministico
0.5 - 0.8 - equilibrato
1.0       - distribuzione originale circa
>1.0      - più imprevedibile

- top_k
considera solamente i K token più probabili
con top_k=50 Se il vocabolario contiene 50_000 token scarti 49_950 campioni e tieni solo i migliori 50 token
Questo riduce il rischio di selezionare token assurdi

- top_p o nucleus sampling
esempio top_p=0.9, prendi il gruppo minimo di token le cui probabilità sommate raggiungono almeno il 90% e poi campioni da quel gruppo

** Modelli base e modelli instrut
Un modello come GPT-2 è essenzialmente addestrato a continuare il testo
Gli dai una frase e lui continua con il (i) token successivo (i).

I modelli: Qwen Instruct, Llama Instruct, Mistral Instruct è invece stato addestrato per seguire istruzioni.
Puoi dirgli: "spiega in 3 righe che cos'è un ERP" e lui spiega
Per costruire assistenti, estrattori strutturati, o agenti, normalmente utilizzi un modello instruct, non un semplice base model.

Esistono poi dei trucchi per evitare i difetti tipici di un LLM
- Repetition Penalty: penalizza i token già generati per forzare il modello a eplorare nuove direzioni semantiche ed evitare loop infiniti.
Con Beam Search esplora diversi rami di generazione in parallelo per trovare la sequenza con la probabilità complessiva più alta.
Max New Tokens definisce il limite fisico della generazione per evitare che il modello produca testi eccessivamene lunghi o fuori controllo.

Una temperatura troppo alta aumenta i rischi di allucinazioni, mentre unatemperatura troppo bassa porta a risposte ripetitive e prive di stile.
L'ingegneria dei parametri di generazione è fondamentale per adatt Trasforma un modello generico in uno strumento specializzato. 

In [3]:
import torch
from transformers import pipeline, AutoTokenizer


# 1. LA PIPELINE: L'astrazione di alto livello
# La funzione pipeline è lo strumento principale della libreria Transformers per eseguire task predefiniti.
# Internamente, essa coordina l'interazione tra tre componenti fondamentali:
# - Il Tokenizer: che prepara i dati in ingresso.
# - Il Modello: che esegue il calcolo delle probabilità statistica.
# - Il Decoder: che trasforma i risultati numerici nuovamente in testo leggibile.
# Utilizziamo 'distilgpt2', una versione alleggerita di GPT-2 progettata per essere veloce ed efficiente.
# Il parametro 'device=0' indica al sistema di utilizzare la prima GPU disponibile (CUDA),
# mentre 'framework="pt"' forza esplicitamente l'uso di PyTorch.
generator = pipeline('text-generation', model='distilgpt2', device=0)

# Definiamo il 'prompt', ovvero la sequenza di parole iniziale che il modello dovrà completare.
# Usiamo l'inglese per ottenere risultati migliori, dato che il modello è stato addestrato principalmente su questa lingua.
prompt_base = "In the heart of the forest, an ancient robot"

print("--- 1. GENERAZIONE STANDARD (PIPELINE) ---")
# Richiamando l'oggetto 'generator', la pipeline esegue l'intero ciclo di vita dell'inferenza:
# 1. Tokenizzazione del prompt.
# 2. Passaggio dei token al modello linguistico.
# 3. Campionamento delle parole successive fino al raggiungimento di 'max_new_tokens'.
# 4. Decodifica dei token generati in testo leggibile.
risultato_base = generator(prompt_base, max_new_tokens=20, truncation=True)
print(risultato_base[0]['generated_text'])


# 2. IL TOKENIZER: La rappresentazione numerica del linguaggio
# I modelli linguistici non comprendono i caratteri alfabetici, ma operano esclusivamente su numeri.
# La classe AutoTokenizer carica la configurazione specifica per il modello selezionato ('distilgpt2').
# Questo garantisce che il testo venga scomposto esattamente come il modello si aspetta.
tokenizer = AutoTokenizer.from_pretrained('distilgpt2')

print("\n--- 2. DIETRO LE QUINTE: TOKENIZZAZIONE ---")
# Il metodo 'encode' trasforma la stringa in una sequenza di identificativi numerici (ID).
# Ogni numero rappresenta un'unità minima di significato presente nel vocabolario del modello.
tokens = tokenizer.encode(prompt_base)

print(f"Testo originale: {prompt_base}")
print(f"Equivalenti numerici (ID): {tokens}")

# Eseguiamo un ciclo per decodificare singolarmente ogni ID e vedere come il tokenizer ha "spezzato" la frase.
# Noterete che la punteggiatura e gli spazi iniziali vengono spesso accorpati alle parole.
print(f"Scomposizione dei singoli token: {[tokenizer.decode([t]) for t in tokens]}")


# 3. I PARAMETRI DI GENERAZIONE: Controllo della logica e della creatività
# Quando il modello genera una parola, assegna una probabilità a ogni termine del suo vocabolario.
# Possiamo influenzare questo processo di selezione attraverso specifici parametri chiamati iperparametri.

print("\n--- 3. IL BILANCIAMENTO DELLA GENERAZIONE ---")

# ESEMPIO A: Bassa Temperatura (Massima coerenza)
# La 'temperature' controlla quanto il modello debba essere "audace".
# Con un valore basso (0.1), il modello sceglie quasi esclusivamente le parole più probabili.
# Risultato: Testo molto logico e strutturato, ma potenzialmente ripetitivo.
output_freddo = generator(
    prompt_base, 
    max_new_tokens=30, 
    temperature=0.1,  # Valore vicino allo zero: approccio quasi deterministico
    do_sample=True,
    truncation=True
)

# ESEMPIO B: Alta Temperatura e Top-P (Creatività controllata)
# Una temperatura elevata (1.8) "appiattisce" le probabilità, rendendo parole rare molto più probabili.
# Per evitare che il testo diventi del tutto incoerente, usiamo il 'top_p' (Nucleus Sampling).
# Il parametro 'top_p=0.85' dice al modello di considerare solo le parole che sommate costituiscono l'85% 
# della massa di probabilità, eliminando la "coda" di termini senza senso.
output_caldo = generator(
    prompt_base, 
    max_new_tokens=30, 
    temperature=1.8,  # Alta creatività, rischio di instabilità
    top_p=0.85,       # Filtro per mantenere una coerenza grammaticale
    do_sample=True,
    truncation=True
)

print(f"APPROCCIO LOGICO (T=0.1): {output_freddo[0]['generated_text']}")
print(f"APPROCCIO CREATIVO (T=1.8): {output_caldo[0]['generated_text']}")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 2951.82it/s]
[transformers] Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- 1. GENERAZIONE STANDARD (PIPELINE) ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In the heart of the forest, an ancient robot has been discovered by a group of scientists at the University of St Petersburg.







[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- 2. DIETRO LE QUINTE: TOKENIZZAZIONE ---
Testo originale: In the heart of the forest, an ancient robot
Equivalenti numerici (ID): [818, 262, 2612, 286, 262, 8222, 11, 281, 6156, 9379]
Scomposizione dei singoli token: ['In', ' the', ' heart', ' of', ' the', ' forest', ',', ' an', ' ancient', ' robot']

--- 3. IL BILANCIAMENTO DELLA GENERAZIONE ---


[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


APPROCCIO LOGICO (T=0.1): In the heart of the forest, an ancient robot was discovered in the forest.
























APPROCCIO CREATIVO (T=1.8): In the heart of the forest, an ancient robot had gone through its death when its tail flew.
The robot fell backwards, into a hole to create new tracks which led one to take a different
